# Step 2: Synthetic Funnel & A/B Test Data Generation Engine

**Objective:** Programmatically generate user clickstream logs (`events` table) and randomized experiment assignments (`ab_test_assignment` table) using `Faker` and `numpy`.

**Key Design Decisions:**
1. **Relational Link:** We fetch existing `customer_unique_id`s from our MySQL Olist database to ensure logical integrity across real and synthetic datasets.
2. **Funnel Sequence:** Each user undergoes a logical progression: `signup` $\rightarrow$ `activated` $\rightarrow$ `purchased` (with natural drop-off rates at each stage).
3. **Baked-In Experiment Effect:** Users in the `treatment` group receive a controlled **~8–10% conversion rate lift** over the `control` group so our statistical hypothesis test discovers a true effect.

In [1]:
import os
import random
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
from faker import Faker
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus

# Initialize Faker with seed for reproducibility
fake = Faker()
Faker.seed(42)
np.random.seed(42)
random.seed(42)

# Load database credentials
load_dotenv(override=True)

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', '127.0.0.1').strip()
DB_PORT = os.getenv('DB_PORT', '3306').strip()
DB_NAME = os.getenv('DB_NAME')

encoded_password = quote_plus(DB_PASSWORD)
connection_string = f"mysql+pymysql://{DB_USER}:{encoded_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(connection_string, pool_pre_ping=True)

# Fetch customer IDs from the real Olist table we uploaded in Step 1
print("Fetching customer IDs from MySQL...")
olist_customers = pd.read_sql("SELECT DISTINCT customer_unique_id FROM customers;", con=engine)
customer_pool = olist_customers['customer_unique_id'].tolist()

print(f"✅ Successfully fetched {len(customer_pool):,} customer IDs from Olist MySQL database!")

Fetching customer IDs from MySQL...
✅ Successfully fetched 96,096 customer IDs from Olist MySQL database!


## 1. Generating User Profiles & A/B Experiment Assignment
We assign each customer to an acquisition channel (`Direct`, `Organic Search`, `Paid Search`, `Social Media`, `Email`) and randomly assign them 50/50 to either the **Control** or **Treatment** group.

In [2]:
# Number of users to generate
NUM_USERS = len(customer_pool)

acquisition_channels = ['Organic Search', 'Paid Search', 'Social Media', 'Direct', 'Email']
channel_weights = [0.35, 0.25, 0.20, 0.10, 0.10]  # Realistic channel distribution

user_records = []
ab_assignment_records = []

# Date range for signups (spanning 2017 to mid-2018 to match Olist timeline)
start_date = datetime(2017, 1, 1)
end_date = datetime(2018, 6, 30)

for user_id in customer_pool:
    # 1. Signup Metadata
    signup_time = fake.date_time_between(start_date=start_date, end_date=end_date)
    channel = np.random.choice(acquisition_channels, p=channel_weights)
    
    # 2. A/B Test Group Assignment (50/50 split)
    ab_group = 'treatment' if random.random() > 0.5 else 'control'
    
    user_records.append({
        'user_id': user_id,
        'signup_timestamp': signup_time,
        'acquisition_channel': channel
    })
    
    ab_assignment_records.append({
        'user_id': user_id,
        'group_assignment': ab_group
    })

users_df = pd.DataFrame(user_records)
ab_assignments_df = pd.DataFrame(ab_assignment_records)

print(f"Users generated: {len(users_df):,}")
print("\nA/B Group Distribution:")
print(ab_assignments_df['group_assignment'].value_counts(normalize=True))
ab_assignments_df.head()

Users generated: 96,096

A/B Group Distribution:
group_assignment
treatment    0.500406
control      0.499594
Name: proportion, dtype: float64


,user_id,group_assignment
0,861eff4711a542e4b93843c6dd7febb0,treatment
1,290c77bc529b7ac935b93aa66c333dc3,control
2,060e732b5b29e8181a18229c7b0b2b5e,control
3,259dac757896d24d7702b9acbbff3f3c,control
4,345ecd01c38d18a9036ed96c73b8d066,treatment


## 2. Generating Funnel Events Stream (`events` Table)
Here we simulate realistic conversion dynamics:
* **Stage 1 (Signup):** 100% of generated users sign up.
* **Stage 2 (Activation):** ~60% of signed-up users activate their profile / browse catalog.
* **Stage 3 (Purchase Conversion):** 
  * **Control Group:** ~15.0% purchase conversion rate from activation.
  * **Treatment Group:** ~16.5% purchase conversion rate from activation (~10% relative lift).

In [3]:
event_records = []
ab_outcomes = []

# Conversion Probabilities
ACTIVATION_RATE = 0.60         # 60% of signups activate
CONTROL_PURCHASE_RATE = 0.150  # 15.0% of activated control users purchase
TREATMENT_PURCHASE_RATE = 0.165 # 16.5% of activated treatment users purchase (10% relative lift)

# Map assignments for fast lookup
group_lookup = dict(zip(ab_assignments_df['user_id'], ab_assignments_df['group_assignment']))

for idx, row in users_df.iterrows():
    u_id = row['user_id']
    t_signup = row['signup_timestamp']
    group = group_lookup[u_id]
    
    # Event 1: Signup (Always occurs)
    event_records.append({
        'user_id': u_id,
        'event_type': 'signup',
        'event_timestamp': t_signup,
        'acquisition_channel': row['acquisition_channel']
    })
    
    # Event 2: Activated (Conditional)
    is_activated = random.random() < ACTIVATION_RATE
    converted_to_purchase = 0
    
    if is_activated:
        # Activation timestamp is minutes to hours after signup
        t_activated = t_signup + timedelta(minutes=random.randint(5, 120))
        event_records.append({
            'user_id': u_id,
            'event_type': 'activated',
            'event_timestamp': t_activated,
            'acquisition_channel': row['acquisition_channel']
        })
        
        # Event 3: Purchased (Conditional based on Control vs Treatment)
        p_threshold = TREATMENT_PURCHASE_RATE if group == 'treatment' else CONTROL_PURCHASE_RATE
        is_purchased = random.random() < p_threshold
        
        if is_purchased:
            converted_to_purchase = 1
            # Purchase timestamp is hours to days after activation
            t_purchased = t_activated + timedelta(hours=random.randint(1, 48))
            event_records.append({
                'user_id': u_id,
                'event_type': 'purchased',
                'event_timestamp': t_purchased,
                'acquisition_channel': row['acquisition_channel']
            })
            
    # Save outcome record for A/B testing dataframe
    ab_outcomes.append(converted_to_purchase)

# Add outcome column to A/B assignment dataframe
ab_assignments_df['converted'] = ab_outcomes
events_df = pd.DataFrame(event_records)

print(f"Total Events Generated: {len(events_df):,}")
print("\nEvent Counts by Stage:")
print(events_df['event_type'].value_counts())

Total Events Generated: 162,680

Event Counts by Stage:
event_type
signup       96096
activated    57555
purchased     9029
Name: count, dtype: int64


## 3. Verify Simulated Effect & Push Synthetic Tables to MySQL
Let's quick-check the observed conversion rates between Control and Treatment before exporting to MySQL.

In [4]:
# Verify Conversion Rates
cr_summary = ab_assignments_df.groupby('group_assignment')['converted'].agg(['count', 'mean'])
cr_summary['mean'] = cr_summary['mean'].apply(lambda x: f"{x:.2%}")
print("Simulated Conversion Summary:")
print(cr_summary)

# Push to MySQL
print("\nExporting synthetic tables to MySQL `ecommerce_db`...")

events_df.to_sql('events', con=engine, if_exists='replace', index=False)
ab_assignments_df.to_sql('ab_test_assignment', con=engine, if_exists='replace', index=False)

print("🚀 Step 2 Complete: `events` and `ab_test_assignment` successfully loaded into MySQL!")

Simulated Conversion Summary:
                  count    mean
group_assignment               
control           48009   8.77%
treatment         48087  10.03%

Exporting synthetic tables to MySQL `ecommerce_db`...
🚀 Step 2 Complete: `events` and `ab_test_assignment` successfully loaded into MySQL!
